# Amazon ESCI 日本語検索 — XGBoost Learning to Rank

## 結論と実装の概要

`amazon-jp-v2` の `content_low` lexical検索を第一段階とし、その上位100件をOpenSearch LTRプラグイン内のXGBoost LambdaMARTで再順位する第二段階を実装した。学習・選択にはTask 1 JP trainのみを使い、query_id単位SHA-1で固定した20%をvalidation、test 3,123 queryはモデル選択完了後の最終評価に一度だけ使う。

- 学習単位: `(query_id, product_id)`。同一queryをgroupとしてXGBoostへ渡す。
- 教師ラベル: 学習用順序 `I=0, S=1, C=2, E=3`。評価はAmazon ESCIのgain `I=0, S=.01, C=.1, E=1`。
- 候補: 公式Task 1評価では提供候補すべて。実運用評価では`content_low`のcorpus top 100。
- モデル選択: `rank:pairwise` / `rank:ndcg` と木深さ3 / 5をtrain内validationで比較。early stoppingで木数を決定。
- 配備: `esci_jp_features_v2` と `esci_jp_xgb_v2` を `.ltrstore` に登録済み。OpenSearchの`rescore`で推論する。

### 重要な実装上の注意

OpenSearch LTR 3.7のlegacy XGBoost evaluatorは、非match特徴を`NaN`にする一方、XGBoost JSONの`missing`分岐を推論に反映しない。そこで各feature queryへboost `1e-6` のmatch-all節を加え、非一致も明示的な極小値として記録・学習・推論する。さらにLucene scoreは負値を許さないため、各木の全leafへ同じ定数を加える。どちらも順位を変えず、Notebook側とOpenSearch側の推論を一致させる。

## 特徴量

全特徴量はOpenSearch自身が計算する。同じfeature setから学習ログとオンライン推論を作るため、analyzer・BM25・phrase挙動のNotebook再実装によるずれを避けている。

|分類|特徴|意図|
|---|---|---|
|title|`title_bm25`, `title_phrase`|通常一致と語順を伴う強い一致|
|brand|`brand_bm25`, `brand_phrase`|ブランド指名query|
|表記揺れ|`title_reading`, `brand_reading`|カナ・英字・読みの差|
|部分一致|`title_ngram`, `brand_ngram`|型番、連結語、表記断片|
|補助文|`bullet_bm25`, `description_bm25`|用途・属性語|
|強いbaseline|`lexical_v2`|前段で選択済みの`content_low` query score|

この段階では価格・人気など時点依存情報を入れていない。リークなしの履歴特徴を用意できた段階で追加する。

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ARTIFACT_DIR = Path("artifacts/ltr_xgboost")
metrics = json.loads((ARTIFACT_DIR / "metrics.json").read_text())
tuning = pd.read_csv(ARTIFACT_DIR / "validation_tuning.csv")
importance = pd.read_csv(ARTIFACT_DIR / "feature_importance.csv")
parity = pd.read_csv(ARTIFACT_DIR / "parity_check.csv")
end_to_end = pd.read_csv(ARTIFACT_DIR / "end_to_end_test.csv")
print("artifacts loaded:", ARTIFACT_DIR.resolve())

artifacts loaded: /Users/masamichiimaseki/Documents/opensearch-trial/amazon/artifacts/ltr_xgboost


## 1. train内validationによるモデル選択

既存のlexical tuningと同じ分割式 `int(SHA1(query_id), 16) % 5 == 0` をvalidationに使う。5,844 queryでfitし、1,440 queryでobjective・depth・木数を選ぶ。testをここでは参照しない。

In [2]:
display(tuning.round({"validation_official_nDCG": 6}))
best = tuning.iloc[0]
display(Markdown(f"**選択:** `{best['config']}` / {int(best['trees'])} trees（validation公式nDCG `{best['validation_official_nDCG']:.6f}`）"))

,config,objective,max_depth,trees,validation_official_nDCG
0,ndcg_d5,rank:ndcg,5,28,0.845897
1,pairwise_d3,rank:pairwise,3,142,0.845080
2,ndcg_d3,rank:ndcg,3,66,0.844809
3,pairwise_d5,rank:pairwise,5,47,0.844090


**選択:** `ndcg_d5` / 28 trees（validation公式nDCG `0.845897`）

## 2. held-out test — 公式Task 1候補集合

提供された全候補を順位付けし、未判定を新たに作らないAmazon ESCI Task 1形式。score同値は`product_id`昇順で固定する。`content_low` baselineは`esci_lexical_v2.ipynb`の0.84744を再現する。

In [3]:
candidate = pd.DataFrame([
    {"system": "content_low baseline", "official nDCG": metrics["test_baseline_official_nDCG"]},
    {"system": "XGBoost LTR", "official nDCG": metrics["test_xgboost_official_nDCG"]},
])
display(candidate.round({"official nDCG": 6}))
p = metrics["candidate_paired"]
display(Markdown(
    f"**差分 {p['delta']:+.6f}**、paired bootstrap 95% CI "
    f"[{p['ci95_low']:+.6f}, {p['ci95_high']:+.6f}]、"
    f"{p['wins']:,}勝 / {p['ties']:,}同順位 / {p['losses']:,}敗。"
))

,system,official nDCG
0,content_low baseline,0.847443
1,XGBoost LTR,0.849315


**差分 +0.001872**、paired bootstrap 95% CI [-0.000429, +0.004176]、1,386勝 / 446同順位 / 1,291敗。

## 3. held-out test — corpus top-100再順位

実運用に近い評価。前段`content_low`が全339,059商品から返したtop 100だけをLTRで並べ替える。corpus検索ではtop 100中の未判定商品をgain 0として扱うため、これはTask 1公式候補集合nDCGとは別指標である。再順位は候補集合を変えないのでRecall@100は不変になる。

In [4]:
e2e = (end_to_end.groupby("variant")
       .agg(nDCG_at_10=("nDCG_at_10_unjudged0", "mean"),
            nDCG_at_100=("nDCG_at_100_unjudged0", "mean"),
            Recall_at_100=("recall_at_100_judged_positive", "mean"),
            judged_at_10=("judged_at_10", "mean")))
display(e2e.round(6))
p = metrics["end_to_end_paired"]
display(Markdown(
    f"nDCG@100差分 **{p['delta']:+.6f}**、paired bootstrap 95% CI "
    f"[{p['ci95_low']:+.6f}, {p['ci95_high']:+.6f}]。"
))

,nDCG_at_10,nDCG_at_100,Recall_at_100,judged_at_10
variant,,,,
baseline,0.394185,0.470217,0.484678,0.425328
xgboost,0.401474,0.474418,0.484678,0.434614


nDCG@100差分 **+0.004200**、paired bootstrap 95% CI [+0.001801, +0.006679]。

## 4. 特徴量重要度

XGBoostのgain重要度。因果的な寄与ではなく、木がvalidationで選ばれた条件下でどの特徴を分岐に使ったかの診断値として読む。

In [5]:
display(importance.round({"gain": 4}))

,feature,gain
0,lexical_v2,0.5902
1,title_reading,0.1115
2,title_ngram,0.0727
3,title_phrase,0.0481
4,brand_phrase,0.0353
5,description_bm25,0.0284
6,title_bm25,0.0271
7,brand_ngram,0.0243
8,bullet_bm25,0.0217
9,brand_bm25,0.0212


## 5. OpenSearch内推論とのparity

XGBoostのraw marginへleaf shiftの定数を加えた値と、OpenSearch LTR rescoreのscoreを3 queryで比較する。順位一致を必須条件とした。

In [6]:
display(pd.DataFrame([{
    "max absolute score difference": metrics["parity_max_abs_diff"],
    "rank exactly equal": metrics["parity_rank_equal"],
    "checked rows": len(parity),
}]))
display(parity[["query_id", "product_id", "python_score", "opensearch_score", "abs_diff"]].head(10))

,max absolute score difference,rank exactly equal,checked rows
0,1.483215e-07,True,96


,query_id,product_id,python_score,opensearch_score,abs_diff
0,438,B08PBJSJM8,0.440430,0.440430,6.447678e-08
1,438,B087R9X5X7,0.422002,0.422002,3.171127e-08
2,438,B0881K8XHS,0.522889,0.522889,1.221619e-09
3,438,B089VZ3RY8,0.468114,0.468114,5.819923e-08
4,438,B08H142Z8H,0.478740,0.478740,3.212868e-08
5,438,B08H163Q5G,0.478740,0.478740,3.212868e-08
6,438,B08K35ZN26,0.523527,0.523527,2.815323e-08
7,438,B093LDN6JM,0.512369,0.512369,2.326965e-10
8,438,B086DQYLXH,0.473068,0.473068,5.795105e-08
9,438,B093LFS7PX,0.604706,0.604706,5.777939e-08


## 6. 配備と再実行

登録済み名称はfeature set `esci_jp_features_v2`、model `esci_jp_xgb_v2`。検索bodyでは第一段階queryに次を加える。

```json
{
  "rescore": {
    "window_size": 100,
    "query": {
      "rescore_query": {
        "sltr": {"model": "esci_jp_xgb_v2", "params": {"keywords": "ユーザークエリ"}}
      },
      "query_weight": 0.0,
      "rescore_query_weight": 1.0
    }
  }
}
```

再学習・特徴量収集・登録・test評価はリポジトリrootから `.venv/bin/python amazon/run_ltr_xgboost.py`。feature DSLとXGBoost JSON変換は`ltr_features.py`に共通化した。

## 判断

end-to-end nDCG@100の95% CIはゼロを跨がず、第二段階を続ける根拠は得られた。一方、公式候補集合nDCGのCIはゼロを跨ぐ。次はrescore window（50/100/200）のvalidation選択、query群別の悪化分析、価格・カテゴリ等のリークしない構造化特徴を優先する。semantic特徴は同じ候補固定評価へ追加し、Recall改善は別途hybrid candidate generationとして評価する。